# LightForge GAME: free GPU qualification

**Prepared research experiment; no GAME GPU result or 75% speedup claim is included.** This companion tests the original five GAME graphs, all eight diffusion steps and all 12 inference calls on one 14-second passage. Four runtime/optimization arms each execute plain, raw-capture and profiling observers.

The input is the **public demo mixture**, averaged to mono. It is not a separated vocal stem, so this diagnostic cannot establish production vocal quality, the complete vocal pipeline, Android behavior or whole-analysis speedup. The 75% target requires the same complete analysis to finish in one quarter of its baseline time.

## Choose a free runtime

- **Colab:** Runtime → Change runtime type → an available free GPU (T4 preferred), then connect. Do not select a paid upgrade.
- **Kaggle:** import this notebook, enable Internet and select an available free GPU. Account verification or remaining quota may be required.
- Run one interactive notebook session. There is no app server, tunnel, automatic reconnect or quota workaround.

Initial downloads are approximately 3.6 GB. Allow **12 GiB free disk** for the original models, isolated runtimes and evidence. Only the public release APK and public model/runtime sources are downloaded; no private audio, Drive mount, API token or signing key is needed.

Sources: [Colab limits](https://research.google.com/colaboratory/faq.html), [Kaggle notebooks](https://www.kaggle.com/docs/notebooks), [ORT CUDA requirements](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html), [cuDNN 9.10.2 support](https://docs.nvidia.com/deeplearning/cudnn/backend/v9.10.2/reference/support-matrix.html).

## 1. Configure an isolated workspace

The GAME harness and helpers are frozen to commit `d9b42bb79145cfe81367fd70e2ca0ff7f4d52c12`. Python3.11+, Linux x86-64 and Node18+ are required. Each run gets a new evidence directory; previous evidence is preserved.

In [ ]:
import datetime, hashlib, importlib.util, json, os, platform, shutil, subprocess, sys
import re, signal, struct, wave, tarfile, tempfile, urllib.request, uuid, zipfile
from pathlib import Path

SOURCE_COMMIT = "d9b42bb79145cfe81367fd70e2ca0ff7f4d52c12"
REPOSITORY = "https://github.com/CyberBASSLord-666/LightForge.git"
BASE = Path("/kaggle/working" if Path("/kaggle/working").is_dir() else "/content" if Path("/content").is_dir() else Path.cwd())
WORK = BASE / "lightforge-game-gpu-qualification"
REPO, TOOLCHAIN, ASSETS = (WORK / name for name in ("source", "toolchain", "assets"))
RUN = WORK / "evidence" / (datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8])
assert sys.version_info >= (3, 11), "Use a Python 3.11+ runtime."
assert hasattr(tarfile, "data_filter"), "This Python needs security updates supporting tarfile's data filter."
assert platform.system() == "Linux" and platform.machine() == "x86_64", "Linux x86-64 is required."
assert re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT), "Expected a complete immutable source commit pin."
WORK.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=False)
assert shutil.disk_usage(WORK).free >= 12 * 1024**3, "Need at least 12 GiB free disk."
print("Evidence directory:", RUN)
print("Python:", sys.version.split()[0], "Platform:", platform.platform())
node_version = subprocess.run(["node", "--version"], check=True, capture_output=True, text=True).stdout.strip()
assert int(node_version.lstrip("v").split(".")[0]) >= 18, "Node18+ is required for the production source proof."
print("Node:", node_version)

## 2. Check the assigned GPU

Record the actual GPU and driver before large downloads. Detecting a GPU does not prove that a model ran there; the graph traces later establish CUDA placement and expose CPU fallback.

In [ ]:
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True, timeout=30)
(RUN / "gpu-inventory.txt").write_text(gpu.stdout + gpu.stderr)
assert gpu.returncode == 0 and gpu.stdout.strip(), "No usable NVIDIA GPU; select a free GPU runtime."
for line in gpu.stdout.strip().splitlines():
    name, driver, memory, capability = [value.strip() for value in line.split(",")]
    assert tuple(map(int, driver.split("."))) >= (525, 60, 13), "The pinned CUDA 12 stack needs a compatible driver."
    assert float(capability) >= 6.0, "The pinned cuDNN runtime requires compute capability 6.0 or later."
print(gpu.stdout.strip())
print("Inventory only; GPU execution and numerical equivalence remain untested.")

## 3. Retrieve the exact experiment source

A fixed commit provides production `NativeGame.java`, the GAME harness, original graph manifest and JavaScript source used to prove the input clock. No application source is edited.

In [ ]:
def command(arguments, **kwargs):
    return subprocess.run([str(value) for value in arguments], check=True, text=True, **kwargs)

patterns = ["/android/src/", "/android/native-runtime.json", "/tests/NativeGameTest.java",
            "/tools/benchmark_game_accelerator.py", "/tools/game_benchmark/",
            "/tools/benchmark_deux_accelerator.py", "/tools/profile_deux_operators.py",
            "/tools/benchmark_deux_execution.py", "/tests/NativeDeuxExecutionBenchmark.java",
            "/tools/bootstrap_toolchain.py", "/web/analysis/game.js", "/web/analysis/wav-reader.js",
            "/web/analysis/models/game/manifest.json", "/web/analysis/models/game/config.json",
            "/web/analysis/models/deux/manifest.json"]
if not (REPO / ".git").exists():
    REPO.mkdir(parents=True, exist_ok=True)
    command(["git", "init", "-q", REPO])
    command(["git", "-C", REPO, "remote", "add", "origin", REPOSITORY])
assert command(["git", "-C", REPO, "remote", "get-url", "origin"], capture_output=True).stdout.strip() == REPOSITORY
head_check = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True)
if head_check.returncode != 0:
    # A first fetch may have been interrupted; retry into this empty checkout.
    command(["git", "-C", REPO, "config", "core.sparseCheckout", "true"])
    command(["git", "-C", REPO, "config", "remote.origin.promisor", "true"])
    command(["git", "-C", REPO, "config", "remote.origin.partialclonefilter", "blob:none"])
    (REPO / ".git/info/sparse-checkout").write_text("\n".join(patterns) + "\n")
    command(["git", "-C", REPO, "fetch", "--filter=blob:none", "--depth=1", "origin", SOURCE_COMMIT])
    command(["git", "-C", REPO, "checkout", "--detach", SOURCE_COMMIT])
head = command(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True).stdout.strip()
assert head == SOURCE_COMMIT, "Existing checkout differs; choose a new WORK path and rerun setup."
assert not command(["git", "-C", REPO, "status", "--porcelain", "--untracked-files=no"], capture_output=True).stdout.strip(), "Tracked source changed; choose a new WORK path."
print("Verified source commit:", head)

## 4. Define verified downloads

Streamed downloads must match their declared sizes and SHA-256 digests before being installed. Cached complete files are rehashed; a partial or changed file cannot become an accepted dependency.

In [ ]:
def digest(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def fetch(url, destination, size, sha256):
    destination = Path(destination)
    if destination.is_file() and not destination.is_symlink() and destination.stat().st_size == size and digest(destination) == sha256:
        print("Verified cached", destination.name)
        return destination
    assert not destination.is_symlink(), "Refusing a symlink destination."
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".partial")
    assert not partial.is_symlink(), "Refusing a symlink partial download."
    request = urllib.request.Request(url, headers={"User-Agent": "LightForge-public-GPU-experiment/1"})
    with urllib.request.urlopen(request, timeout=180) as response, partial.open("wb") as output:
        count = 0
        while block := response.read(1024 * 1024):
            count += len(block)
            if count > size:
                raise ValueError("Download exceeds pinned size: " + destination.name)
            output.write(block)
    assert partial.stat().st_size == size and digest(partial) == sha256, "Download digest mismatch: " + destination.name
    partial.replace(destination)
    print("Verified download", destination.name)
    return destination

def extract_member(archive, member, destination, size=None, sha256=None):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    assert not destination.is_symlink(), "Refusing a symlink destination."
    partial = destination.with_name(destination.name + ".partial")
    assert not partial.is_symlink(), "Refusing a symlink partial output."
    with archive.open(member) as source, partial.open("wb") as output:
        shutil.copyfileobj(source, output, 1024 * 1024)
    if size is not None:
        assert partial.stat().st_size == size, "Extracted size mismatch: " + member
    if sha256 is not None:
        assert digest(partial) == sha256, "Extracted digest mismatch: " + member
    partial.replace(destination)
    return destination

## 5. Recover original GAME models and public audio

The public 2.3.1 APK is an asset container only. Its whole-file digest is fixed; every GAME graph and its config must also match the frozen source manifest. The original model manifest retains the author and model license. Nothing is rebuilt, signed, installed or published.

In [ ]:
APK_SIZE = 1205116958
APK_SHA = "af83bf403875c55d42fd695d43f6e193114899c1324fbaeaffd6c02d299d882f"
AUDIO_SHA = "33f07d502ba19832b62b97d8fb4a11354e81310264bd885d7a06438228773650"
MANIFEST_SHA = "51e172cfaa967d9e2518f01f508a64d49cd283d23ae8e8456af9c6e76eeb4f97"
APK_URL = "https://github.com/CyberBASSLord-666/LightForge/releases/download/v2.3.1/LightForge-2.3.1.apk"
apk = fetch(APK_URL, ASSETS / "LightForge-2.3.1.apk", APK_SIZE, APK_SHA)
manifest_path = REPO / "web/analysis/models/game/manifest.json"
assert digest(manifest_path) == MANIFEST_SHA
manifest = json.loads(manifest_path.read_text())
MODELS, AUDIO = ASSETS / "game", ASSETS / "glass-castle.wav"
MODELS.mkdir(exist_ok=True)
with zipfile.ZipFile(apk) as archive:
    assert hashlib.sha256(archive.read("assets/analysis/models/game/manifest.json")).hexdigest() == MANIFEST_SHA
    for name, pin in manifest["files"].items():
        assert Path(name).name == name, "Unexpected model path."
        extract_member(archive, "assets/analysis/models/game/" + name, MODELS / name, pin["bytes"], pin["sha256"])
    extract_member(archive, "assets/demo/glass-castle.wav", AUDIO, sha256=AUDIO_SHA)
shutil.copyfile(manifest_path, MODELS / "manifest.json")
assert len(manifest["files"]) == 6 and len(list(MODELS.glob("*.onnx"))) == 5
print("Verified five original GAME graphs, config, manifest and public demo.")

## 6. Prepare the exact first passage

The 64-second demo is stereo signed PCM16 at 44.1 kHz. For each sample, decode left and right as `sample / 32768`, average them and store little-endian Float32. No resampling, gain change, model conversion or vocal separation occurs.

Python prepares samples **0–617,399** (14 seconds). Independently, the next cell uses the actual production WAV reader and JavaScript Float32 arithmetic and requires byte-for-byte agreement.

In [ ]:
with wave.open(str(AUDIO), "rb") as source:
    assert source.getnchannels() == 2 and source.getsampwidth() == 2
    assert source.getframerate() == 44100 and source.getcomptype() == "NONE"
    TOTAL_SAMPLES = source.getnframes()
    assert TOTAL_SAMPLES == 2822400, "Public fixture clock differs."
    PASSAGE_SAMPLES = 14 * 44100
    raw = source.readframes(PASSAGE_SAMPLES)
assert len(raw) == PASSAGE_SAMPLES * 4
PCM = RUN / "public-demo-mixture-first14s.f32"
with PCM.open("xb") as output:
    for left, right in struct.iter_unpack("<hh", raw):
        output.write(struct.pack("<f", (left / 32768.0 + right / 32768.0) * 0.5))
assert PCM.stat().st_size == PASSAGE_SAMPLES * 4
print("Diagnostic mixture samples:", PASSAGE_SAMPLES, "PCM SHA-256:", digest(PCM))
assert digest(PCM) == "a75b3b8d59c87d428e38e45db5f54ad8d2b1613fe79ee07c2595fecbf9215274", "Public first-passage bytes differ from the independently recovered fixture."

## 7. Prove the input bytes and production window plan

The Node check invokes unchanged `WavReader.stereo44100` for the actual public audio, then averages its Float32 channels. It also invokes the unchanged GAME `process()` method with synthetic nonzero planning buffers and a recording native callback. That second check proves all six windows and seeds against independently calculated Python values; it performs **no model inference** and establishes **no musical result**.

In [ ]:
NODE_INPUT_PROOF = r"""const fs = require('node:fs');
const path = require('node:path');
const crypto = require('node:crypto');
const assert = require('node:assert/strict');
const [root, audioPath, pcmPath, proofPath] = process.argv.slice(1);
const WavReader = require(path.join(root, 'web/analysis/wav-reader.js'));
const GAME = require(path.join(root, 'web/analysis/game.js'));
const audio = fs.readFileSync(audioPath);
const manifest = JSON.parse(fs.readFileSync(path.join(root, 'web/analysis/models/game/manifest.json')));
const hash = data => crypto.createHash('sha256').update(data).digest('hex');
(async()=>{
  globalThis.fetch = async (url, options={}) => {
    if (String(url) === 'https://lightforge-public-fixture.invalid/manifest.json') return {json:async()=>manifest};
    assert.equal(String(url), 'https://lightforge-public-fixture.invalid/demo.wav');
    const match = /^bytes=(\d+)-(\d+)$/.exec(options.headers.Range);
    assert(match); const first=Number(match[1]), last=Math.min(Number(match[2]),audio.length-1);
    return new Response(audio.subarray(first,last+1), {status:206, headers:{
      'Content-Length':String(last-first+1), 'Content-Range':`bytes ${first}-${last}/${audio.length}`}});
  };
  const reader = new WavReader('https://lightforge-public-fixture.invalid/demo.wav');
  await reader.open();
  assert.equal(reader.rate,44100); assert.equal(reader.samples,2822400);
  const stereo = await reader.stereo44100(0,14*44100);
  const mono = new Float32Array(stereo[0].length);
  for(let i=0;i<mono.length;i++) mono[i]=(stereo[0][i]+stereo[1][i])*.5;
  const bytes = Buffer.alloc(mono.length*4);
  for(let i=0;i<mono.length;i++) bytes.writeFloatLE(mono[i],i*4);
  assert(bytes.equals(fs.readFileSync(pcmPath)), 'Python PCM differs from production-reader Float32 mixture');
  const observed=[], reads=[];
  const adapter = await GAME.create({baseUrl:'https://lightforge-public-fixture.invalid/',
    nativeInfer:async(pcm,language,seed)=>{
      observed.push({samples:pcm.length,language,seed}); return [];
    }});
  try {
    await adapter.process(async(first,count)=>{
      reads.push({first,last:first+count}); return new Float32Array(count).fill(.125);
    },reader.samples,{language:0});
  } finally {await adapter.release();}
  assert.equal(reads.length,observed.length);
  const plan=observed.map((value,index)=>({index,...reads[index],...value}));
  fs.writeFileSync(proofPath,JSON.stringify({
    schema:'lightforge.game-notebook-input-proof.v1',
    publicAudioSha256:hash(audio),pcmSha256:hash(bytes),totalSamples:reader.samples,
    sampleRate:44100,selectedPassage:0,byteIdenticalToProductionReaderMix:true,plan,
    planOnly:true,modelInferenceExecuted:false,qualityApproved:false,
    sourceHashes:Object.fromEntries(['web/analysis/wav-reader.js','web/analysis/game.js'].map(name=>[name,hash(fs.readFileSync(path.join(root,name)))]))
  },null,2)+'\n',{flag:'wx'});
})().catch(error=>{console.error(error);process.exitCode=1;});"""
proof_path = RUN / "input-source-proof.json"
with (RUN / "input-source-proof.log").open("x") as log:
    command(["node", "-e", NODE_INPUT_PROOF, REPO, AUDIO, PCM, proof_path],
            stdout=log, stderr=subprocess.STDOUT)
proof = json.loads(proof_path.read_text())
expected_plan = []
for index in range((TOTAL_SAMPLES + 12*44100 - 1) // (12*44100)):
    first = max(0, index*12 - 2) * 44100
    last = min(TOTAL_SAMPLES, (index+1)*12*44100 + 2*44100)
    expected_plan.append(dict(index=index, first=first, last=last, samples=last-first,
                              language=0, seed=(2025+index*104729) & 0xffffffff))
assert proof["plan"] == expected_plan, "Actual GAME source window/seed plan differs."
assert proof["byteIdenticalToProductionReaderMix"] and proof["pcmSha256"] == digest(PCM)
assert proof["plan"][0] == dict(index=0, first=0, last=617400, samples=617400, language=0, seed=2025)
INPUT_PROVENANCE = RUN / "input-provenance.json"
input_provenance = {
    "schema":"lightforge.game-passage-input.v1", "sourceCommit":SOURCE_COMMIT,
    "inputKind":"public-demo-mixture", "separatedVocals":False,
    "publicRelease":"https://github.com/CyberBASSLord-666/LightForge/releases/tag/v2.3.1",
    "apkSha256":APK_SHA, "audioMember":"assets/demo/glass-castle.wav", "audioSha256":AUDIO_SHA,
    "sampleRate":44100, "channels":1, "encoding":"float32-le", "samples":PASSAGE_SAMPLES,
    "pcmSHA256":digest(PCM), "sourceSHA256":AUDIO_SHA, "sourceSamples":TOTAL_SAMPLES,
    "passageIndex":0, "firstSample":0, "lastSample":PASSAGE_SAMPLES,
    "language":0, "seed":2025,
    "derivation":"The full public glass-castle.wav WAV file is identified by sourceSHA256. Its PCM16 left/right samples are divided by32768, averaged in double precision and rounded once to Float32. Samples0..617399 form the first14s mixture passage of the64s source; matches productionWavReader plus JavaScript scalaraverage byte-for-byte. This is mixed audio, not separated vocals.",
    "inputSourceProofSha256":digest(proof_path), "qualityApproved":False, "target75Proven":False,
}
INPUT_PROVENANCE.write_text(json.dumps(input_provenance, indent=2)+"\n")
print("Input proof passed; six production windows checked. Selected first passage:", expected_plan[0])

## 8. Install pinned Java and ONNX runtimes

Dependencies are isolated inside the notebook workspace. Java17, the SDK35 platform JAR and ORT1.25.1 are the same verified pins used by the Deux notebook. No Android application build occurs.

In [ ]:
spec = importlib.util.spec_from_file_location("lightforge_bootstrap", REPO / "tools/bootstrap_toolchain.py")
bootstrap = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bootstrap)
TOOLCHAIN.mkdir(exist_ok=True)
for pin in bootstrap.PACKAGES:
    if pin["name"] not in ("jdk17.tar.gz", "platform-35_r02.zip"):
        continue
    archive_path = fetch(pin["url"], TOOLCHAIN / "downloads" / pin["name"], pin["size"], pin["sha256"])
    if pin["name"] == "platform-35_r02.zip":
        with zipfile.ZipFile(archive_path) as archive:
            extract_member(archive, pin["source"] + "/android.jar", TOOLCHAIN / pin["target"] / "android.jar")
    else:
        # Re-extract verified bytes on each setup; do not trust a stale installed marker.
        with tempfile.TemporaryDirectory(dir=TOOLCHAIN) as temp:
            with tarfile.open(archive_path) as archive:
                archive.extractall(temp, filter="data")
            target = TOOLCHAIN / pin["target"]
            if target.exists():
                shutil.rmtree(target)
            shutil.move(str(Path(temp) / pin["source"]), target)

runtime = json.loads((REPO / "android/native-runtime.json").read_text())
host = runtime["host"]
fetch(host["url"], TOOLCHAIN / "onnx" / host["name"], host["bytes"], host["sha256"])
fetch("https://repo.maven.apache.org/maven2/com/microsoft/onnxruntime/onnxruntime_gpu/1.25.1/onnxruntime_gpu-1.25.1.jar",
      TOOLCHAIN / "onnx/onnxruntime_gpu-1.25.1.jar", 397558371,
      "0a22d140ee2a064944b7ee45b7f7a8deb113f58e9be514da85c6dfbe85262649")
fetch("https://repo.maven.apache.org/maven2/org/json/json/20260719/json-20260719.jar", TOOLCHAIN / "test-json.jar", 90031,
      "c243f45f9590c12694a4142ed3f07fc70dfb71e4daebd05ae234bf92a2da92a6")
command([TOOLCHAIN / "jdk17/bin/java", "-version"])
print("Pinned Java compilation and ORT dependencies ready.")

## 9. Install isolated CUDA12 / cuDNN9

The seven exact NVIDIA wheels below provide the native libraries. Their hashes were verified against official PyPI metadata in the original Deux experiment. System drivers and notebook Python packages are unchanged. Extracted library hashes are saved separately from the libraries actually observed in the Java process.

In [ ]:
CUDA_WHEELS = [('nvidia-cudnn-cu12',
  '9.10.2.21',
  'nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl',
  706758467,
  '949452be657fa16687d0930933f032835951ef0892b37d2d53824d1a84dc97a8',
  'ba/51/e123d997aa098c61d029f76663dedbfb9bc8dcf8c60cbd6adbe42f76d049'),
 ('nvidia-cublas-cu12',
  '12.9.1.4',
  'nvidia_cublas_cu12-12.9.1.4-py3-none-manylinux_2_27_x86_64.whl',
  581242350,
  '453611eb21a7c1f2c2156ed9f3a45b691deda0440ec550860290dc901af5b4c2',
  '77/3c/aa88abe01f3be3d1f8f787d1d33dc83e76fec05945f9a28fbb41cfb99cd5'),
 ('nvidia-cuda-runtime-cu12',
  '12.9.79',
  'nvidia_cuda_runtime_cu12-12.9.79-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl',
  3493179,
  '25bba2dfb01d48a9b59ca474a1ac43c6ebf7011f1b0b8cc44f54eb6ac48a96c3',
  'bc/46/a92db19b8309581092a3add7e6fceb4c301a3fd233969856a8cbf042cd3c'),
 ('nvidia-cuda-nvrtc-cu12',
  '12.9.86',
  'nvidia_cuda_nvrtc_cu12-12.9.86-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl',
  89568129,
  '210cf05005a447e29214e9ce50851e83fc5f4358df8b453155d5e1918094dcb4',
  'b8/85/e4af82cc9202023862090bfca4ea827d533329e925c758f0cde964cb54b7'),
 ('nvidia-nvjitlink-cu12',
  '12.9.86',
  'nvidia_nvjitlink_cu12-12.9.86-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl',
  39748338,
  'e3f1171dbdc83c5932a45f0f4c99180a70de9bd2718c1ab77d14104f6d7147f9',
  '46/0c/c75bbfb967457a0b7670b8ad267bfc4fffdf341c074e0a80db06c24ccfd4'),
 ('nvidia-curand-cu12',
  '10.3.10.19',
  'nvidia_curand_cu12-10.3.10.19-py3-none-manylinux_2_27_x86_64.whl',
  68295626,
  '49b274db4780d421bd2ccd362e1415c13887c53c214f0d4b761752b8f9f6aa1e',
  '31/44/193a0e171750ca9f8320626e8a1f2381e4077a65e69e2fb9708bd479e34a'),
 ('nvidia-cufft-cu12',
  '11.4.1.4',
  'nvidia_cufft_cu12-11.4.1.4-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl',
  200877592,
  'c67884f2a7d276b4b80eb56a79322a95df592ae5e765cf1243693365ccab4e28',
  '95/f4/61e6996dd20481ee834f57a8e9dca28b1869366a135e0d42e2aa8493bdd4')]
CUDA_ROOT = TOOLCHAIN / "cuda-wheels"
assert not CUDA_ROOT.is_symlink(), "Refusing a symlink CUDA directory."
if CUDA_ROOT.exists():
    shutil.rmtree(CUDA_ROOT)  # Recreate only this experiment's isolated native runtime.
for package, version, filename, size, sha256, location in CUDA_WHEELS:
    url = "https://files.pythonhosted.org/packages/" + location + "/" + filename
    wheel = fetch(url, TOOLCHAIN / "downloads" / filename, size, sha256)
    with zipfile.ZipFile(wheel) as archive:
        members = [name for name in archive.namelist() if name.startswith("nvidia/") and "/lib/" in name and ".so" in Path(name).name]
        assert members, "No native libraries found in " + filename
        for member in members:
            relative = Path(member)
            assert not relative.is_absolute() and ".." not in relative.parts
            extract_member(archive, member, CUDA_ROOT / relative)
library_dirs = sorted(str(path) for path in CUDA_ROOT.glob("nvidia/*/lib") if path.is_dir())
assert len(library_dirs) == len(CUDA_WHEELS)
benchmark_env = os.environ.copy()
benchmark_env["LD_LIBRARY_PATH"] = os.pathsep.join(library_dirs + ([os.environ["LD_LIBRARY_PATH"]] if os.environ.get("LD_LIBRARY_PATH") else []))
benchmark_env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
benchmark_env["NVIDIA_TF32_OVERRIDE"] = "0"
assert not any(benchmark_env.get(key) for key in ("JAVA_TOOL_OPTIONS", "_JAVA_OPTIONS", "JDK_JAVA_OPTIONS")), "A Java option injection variable is set; use a clean runtime."
setup_receipt = {
    "schema": "lightforge.game-notebook-setup.v1", "sourceCommit": SOURCE_COMMIT,
    "createdUtc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "python": sys.version, "platform": platform.platform(), "publicApkSha256": APK_SHA,
    "node": node_version, "inputPcmSha256": digest(PCM),
    "inputProvenanceSha256": digest(INPUT_PROVENANCE), "publicAudioSha256": digest(AUDIO), "modelManifestSha256": digest(MODELS / "manifest.json"),
    "cudaWheelPins": [dict(package=p, version=v, filename=f, bytes=s, sha256=h,
                           source="https://pypi.org/project/" + p + "/" + v + "/") for p,v,f,s,h,_ in CUDA_WHEELS],
    "extractedNativeLibraries": {str(path.relative_to(CUDA_ROOT)): {"bytes": path.stat().st_size, "sha256": digest(path)}
                                for path in CUDA_ROOT.rglob("*") if path.is_file()},
    "qualityApproved": False, "target75Proven": False, "releaseAuthorized": False,
}
(RUN / "setup-receipt.json").write_text(json.dumps(setup_receipt, indent=2) + "\n")
print("Verified isolated native libraries:", len(setup_receipt["extractedNativeLibraries"]))

## 10. Verify readiness

The committed GAME harness validates source, input provenance, original model hashes and runtime discovery. Readiness does not prove GPU kernel execution or musical quality.

In [ ]:
BENCHMARK = REPO / "tools/benchmark_game_accelerator.py"
assert digest(BENCHMARK) == "3e4c6f5b7fc36205daa70052223f202d8fffeba466cc726e0cebfba4bacbdd50"
base_arguments = [sys.executable, str(BENCHMARK), "--toolchain", str(TOOLCHAIN), "--models", str(MODELS), "--input", str(PCM), "--input-provenance", str(INPUT_PROVENANCE)]
def run_stage(name, extra):
    output = RUN / name
    assert not output.exists(), "Evidence exists; run setup again for a new run directory."
    with (RUN / (name + ".log")).open("w") as log:
        process = subprocess.Popen(base_arguments + ["--output", str(output)] + extra,
                                   cwd=REPO, env=benchmark_env, stdout=log, stderr=subprocess.STDOUT,
                                   start_new_session=True)
        last_progress = None
        try:
            while process.poll() is None:
                try:
                    process.wait(timeout=20)
                except subprocess.TimeoutExpired:
                    try:
                        partial = json.loads((output / "receipt.json").read_text())
                        progress = (partial.get("status"), len(partial.get("runs", [])))
                    except (FileNotFoundError, json.JSONDecodeError):
                        progress = ("STARTING", 0)
                    if progress != last_progress:
                        print(name, "status:", progress[0], "completed arms/runs:", progress[1], flush=True)
                        last_progress = progress
        except KeyboardInterrupt:
            # Stop the benchmark and its Java children, retaining incomplete evidence.
            try:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            except ProcessLookupError:
                process.wait()
            (RUN / (name + "-interrupted.json")).write_text(json.dumps({
                "status": "INTERRUPTED", "completed": False, "target75Proven": False}) + "\n")
            raise
    receipt_path = output / "receipt.json"
    receipt = json.loads(receipt_path.read_text()) if receipt_path.exists() else {"status": "PROCESS_FAILED_WITHOUT_RECEIPT"}
    print(name, "exit:", process.returncode, "status:", receipt["status"])
    if receipt.get("failure"):
        print(receipt["failure"])
    return receipt

readiness = run_stage("readiness", ["--check-readiness"])
assert readiness["status"] == "PREFLIGHT_READY", "Readiness failed. Preserve the evidence; do not substitute CPU execution."

## 11. Run four arms with three observers each

The intended arms are production CPU/ALL, CPU/BASIC, GPU-package CPU/BASIC and GPU-package CUDA/BASIC. Each uses plain, captured and separately profiled execution. Original Float32 graphs, eight diffusion steps, thresholds, source clock and noise seeds remain unchanged. CUDA uses `use_tf32=0`.

Captured tensors, unrounded notes, provider placement and actual loaded native libraries remain in evidence. Observer disagreement or missing substantive CUDA arithmetic in encoder, segmenter or estimator fails closed. The two small duration/boundary conversion graphs may remain on CPU, and their placement is retained. Numerical differences are retained without automatic quality approval. Single cold-passage timings are diagnostic observations; they are not a 75% result or an accepted whole-pipeline speedup.

In [ ]:
assert readiness["status"] == "PREFLIGHT_READY"
experiment = run_stage("qualification", [])
summary_keys = ("status", "measured", "inputsRecheckedAfterQualification", "observerComparisonsPassed",
                "rawOutputsByteIdenticalAcrossVariants", "qualityApproved", "target75Proven",
                "wholeSongSpeedupProven", "fullVocalStageSpeedupProven", "androidSpeedupProven", "releaseAuthorized")
print(json.dumps({key: experiment.get(key) for key in summary_keys}, indent=2))
print("Completed diagnostic passes:", len(experiment.get("runs", [])), "of 12")
for variant, placement in experiment.get("placement", {}).items():
    print(variant, placement)
for row in experiment.get("crossVariantComparisons", []):
    print(row["reference"], "→", row["candidate"], "raw exact:", row["exactParity"],
          "unrounded notes exact:", row["unroundedNotesIdentical"])
print("Diagnostic only: no repeated timing, approved speed ratio or musical-quality approval.")

## 12. Download evidence before the session ends

Run this after completion, a failed gate or an interruption. The ZIP contains this run's receipts, source/input proofs, logs, model-output captures and public-demo PCM. It excludes model weights, the APK, secrets and private files. Incomplete evidence stays labeled incomplete; it does not become a result. Colab prompts for a download; Kaggle exposes the ZIP in its file browser.

In [ ]:
archive_path = RUN.with_suffix(".zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for path in sorted(RUN.rglob("*")):
        if path.is_file() and not path.is_symlink():
            archive.write(path, arcname=str(path.relative_to(RUN.parent)))
print("Evidence:", archive_path.name, "bytes:", archive_path.stat().st_size)
print("SHA-256:", digest(archive_path))
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(str(archive_path.relative_to(Path.cwd())) if archive_path.is_relative_to(Path.cwd()) else str(archive_path)))
else:
    files.download(str(archive_path))

## What this can decide

This experiment can establish whether the assigned GPU executes the original GAME workload and show its numerical differences under controlled runtime changes. It cannot establish separated-vocal accuracy, the complete vocal stage, full-song timing, production-quality approval or the 75% end-to-end target.

**Validation of this saved template:** notebook format and all 12 code cells compile. The public first 14-second PCM was prepared locally, matched byte-for-byte against the actual production WAV reader and JavaScript Float32 averaging, and passed the committed harness input validator. The independently calculated six-window/seed plan matched actual `game.js` execution. GPU execution, full clean-room downloads and visual rendering in Colab/Kaggle still require the target notebook session. No GPU outputs are fabricated or saved here.